# Clustering y búsqueda semántica de reviews
Pipeline limpio y comentado: genera embeddings con SentenceTransformers, reduce dimensión con UMAP, agrupa con HDBSCAN, etiqueta automáticamente cada cluster, construye un índice FAISS para búsqueda semántica y produce un muestreo estratificado de 20 reviews listo para CSV. Usa las mismas rutas/artefactos que el cuaderno anterior y agrega contexto paso a paso.


## Guía rápida de la estructura
- Configuración y carga de datos.
- Funciones reutilizables (encoding, reducción, clustering, etiquetado, búsqueda).
- Generación de embeddings y reducción de dimensión.
- Clustering + catálogo de clusters con etiquetas legibles.
- Índice FAISS y ejemplo de consulta.
- Búsqueda two-stage por centroides de cluster con fallback a ruido.
- Búsqueda rápida: cluster único + FAISS dentro del cluster.
- Muestreo estratificado de 20 reviews (solo id + review) a CSV.

**Notas**
- `cluster_id = -1` es ruido/noise en HDBSCAN.
- Usa el modelo `Alibaba-NLP/gte-Qwen2-1.5B-instruct`; ajusta rutas/hiperparámetros según necesidad.
- Los nombres de cluster se generan automáticamente con TF-IDF sobre las reviews de cada grupo.


## 0. Configuración e imports
Importamos todas las dependencias en un solo lugar y definimos rutas fijas para que el resto del flujo sea reproducible.


In [1]:
from pathlib import Path
import os
import pickle
import warnings

import numpy as np
import pandas as pd
import umap
import hdbscan
import faiss
import torch

from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer


warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_colwidth", 220)

MODEL_NAME = "Alibaba-NLP/gte-Qwen2-1.5B-instruct"
MODEL_REVISION = "main"  # fija la revision para evitar descargas de nueva version de codigo
FALLBACK_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
LOCAL_FILES_ONLY = os.environ.get("LOCAL_FILES_ONLY", "0") == "1"  # default: permite descargar
ALLOW_REMOTE_DOWNLOAD = os.environ.get("ALLOW_REMOTE_MODEL_DOWNLOAD", "1") == "1"
EMB_EXPECTED_DIM = 1536
DATA_PATH = "df_reviews_articles_with_synth_reviews.csv"
EMB_PATH = "review_embeddings_qwen2.npy"
UMAP_PATH = "umap_model_qwen2.pk1"
EMB_UMAP_PATH = "embeddings_umap_qwen2.npy"
CLUSTER_MODEL_PATH = "cluster_model_qwen2.pk1"
FAISS_INDEX_PATH = "faiss_reviews_qwen2.index"
DF_EMB_PATH = "df_with_embeddings_qwen2.pkl"
DF_CLUSTER_PATH = "df_with_clusters_qwen2.pk1"
RANDOM_STATE = 42

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
HF_HOME = Path(os.environ.get("HF_HOME", "/mnt/c/Users/SPARTAN PC/.cache/huggingface"))
if not HF_HOME.exists():
    HF_HOME = Path.home() / ".cache/huggingface"
HF_HOME.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", str(HF_HOME))
os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "0")
os.environ.setdefault("HF_HUB_OFFLINE", "1" if LOCAL_FILES_ONLY else "0")
os.environ.setdefault("TRANSFORMERS_OFFLINE", os.environ["HF_HUB_OFFLINE"])
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
MODEL_CACHE_DIR = HF_HOME / "hub" / f"models--{MODEL_NAME.replace('/', '--')}"
MODEL_LOCAL_DIR = MODEL_CACHE_DIR  # ruta local donde se descargó el modelo
FALLBACK_MODEL_DIR = HF_HOME / "hub" / f"models--{FALLBACK_MODEL_NAME.replace('/', '--')}"

np.random.seed(RANDOM_STATE)


c:\Users\SPARTAN PC\miniconda3\envs\langchain_env_py311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Carga y preprocesamiento del dataset
Leemos el CSV de reviews. El texto se fuerza a `str` y se permite activar eliminación de duplicados por contenido si quieres reducir tamaño antes de calcular embeddings.


In [2]:
DROP_DUPLICATES = False  # pon True si quieres eliminar textos repetidos antes de calcular embeddings

df_raw = pd.read_csv(DATA_PATH)
df = df_raw.copy()
df["review"] = df["review"].fillna("").astype(str)

if DROP_DUPLICATES:
    before = len(df)
    df = df.drop_duplicates(subset="review").reset_index(drop=True)
    print(f"Eliminadas {before - len(df)} filas duplicadas por texto de review.")

print(f"Total filas: {len(df)}")
df.head()


Total filas: 59458


,t_dat,customer_id,article_id,price,sales_channel_id,count,popularity_score,p_review,review_flag,product_code,...,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc,pop_rank,review_stars,review
0,2019-02-25,,108775015,0.008458,2,10841,0.000341,0.03,True,108775,...,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,0.942985,3,"It's an okay top, the straps are a bit thin but it fits well and is comfortable enough for everyday wear."
1,2018-12-23,fbbc4b14371dac97483160a35dbd93e7a57e292aedbe2cef70940ae205ae887c,108775015,0.007186,2,10841,0.000341,0.03,True,108775,...,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,0.942985,4,"I like this top, it's simple and comfy, the black color is perfect for layering or wearing on its own, just wish it came in more sizes."
2,2019-02-03,4c53008e64aa6c59bf1a2b7025ae4afb98be8bc8b457c00e7cae083027adbf63,108775015,0.008458,2,10841,0.000341,0.03,True,108775,...,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,0.942985,4,"Great value for the price, this strap top is soft and comfortable, the narrow straps are a nice touch, and it pairs well with my favorite jeans."
3,2018-12-12,6bdaa2c45d8f21f24bc42f62b873897dec4e5cc00a69cf8d07862645626a6b79,108775015,0.008458,1,10841,0.000341,0.03,True,108775,...,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,0.942985,5,"I'm obsessed with this strap top, it's so versatile and comfortable, the quality is top-notch considering the price, and it's perfect for hot summer days."
4,2018-12-03,1033afaf7b151c626d26baa254b851a2d15368b43b215e9f368d97c2f67a045a,108775015,0.008034,1,10841,0.000341,0.03,True,108775,...,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,0.942985,5,"This strap top is a staple in my wardrobe, it's incredibly soft, fits perfectly, and the black color is timeless, I highly recommend it."


In [3]:
# cuántos tipos diferentes hay (y opcionalmente listarlos)
unique_types = df_raw["product_type_name"].dropna().unique()
len(unique_types), unique_types

(108,
 array(['Vest top', 'Bra', 'Underwear Tights', 'Leggings/Tights',
        'Trousers', 'Hair clip', 'Umbrella', 'Sweater', 'Socks', 'Unknown',
        'Hoodie', 'Hair/alice band', 'Belt', 'Boots', 'Bikini top',
        'Hair string', 'Swimsuit', 'Skirt', 'Kids Underwear top',
        'T-shirt', 'Pyjama set', 'Dress', 'Sunglasses', 'Gloves',
        'Hat/beanie', 'Cap/peaked', 'Earring', 'Top', 'Blazer',
        'Pyjama jumpsuit/playsuit', 'Swimwear bottom', 'Cardigan',
        'Underwear bottom', 'Jacket', 'Shirt', 'Costumes', 'Robe',
        'Shorts', 'Bodysuit', 'Scarf', 'Coat', 'Other accessories',
        'Polo shirt', 'Slippers', 'Night gown', 'Alice band', 'Straw hat',
        'Tailored Waistcoat', 'Ballerinas', 'Tie', 'Necklace',
        'Pyjama bottom', 'Felt hat', 'Bag', 'Bracelet', 'Watch',
        'Dungarees', 'Swimwear set', 'Underwear body', 'Hat/brim',
        'Flat shoe', 'Jumpsuit/Playsuit', 'Sneakers', 'Sandals', 'Blouse',
        'Wedge', 'Long John', 'Sleeping s

## 2. Funciones utilitarias (encoder, reducción, clustering)
Centralizamos las funciones para que cada paso quede empaquetado:
- `encode_reviews`: usa `SentenceTransformer` en batches.
- `reduce_embeddings`: UMAP para pasar de 1536 dims (Qwen2/GTE) a algo manejable para clustering.
- `cluster_embeddings`: HDBSCAN que encuentra clusters de densidad y marca ruido (-1).


In [4]:
def _resolve_model_dir(path: Path) -> Path | None:
    required = ["config.json", "tokenizer.json"]
    candidates = [path]
    snap_root = path / "snapshots"
    if snap_root.exists():
        candidates.extend(sorted([p for p in snap_root.iterdir() if p.is_dir()]))
    for candidate in candidates:
        if not candidate.exists():
            continue
        has_weights = (candidate / "model.safetensors").exists() or (candidate / "model.safetensors.index.json").exists() or (candidate / "pytorch_model.bin").exists()
        if has_weights and all((candidate / fname).exists() for fname in required):
            return candidate
    return None



def load_encoder(model_name: str = MODEL_NAME) -> SentenceTransformer:
    offline = os.environ.get("HF_HUB_OFFLINE", "0") == "1" or LOCAL_FILES_ONLY
    allow_remote = ALLOW_REMOTE_DOWNLOAD and not offline

    def _dir_for(name: str) -> Path:
        return HF_HOME / "hub" / f"models--{name.replace('/', '--')}"

    target_root = _dir_for(model_name)
    fallback_root = _dir_for(FALLBACK_MODEL_NAME)
    resolved_dir = _resolve_model_dir(target_root)
    fallback_dir = _resolve_model_dir(fallback_root)

    candidates: list[tuple[str, bool, str | None, str]] = []
    if resolved_dir is not None:
        candidates.append((str(resolved_dir), True, None, f"cache:{model_name}"))
    if fallback_dir is not None and (resolved_dir is None or model_name != FALLBACK_MODEL_NAME):
        candidates.append((str(fallback_dir), True, None, f"cache:{FALLBACK_MODEL_NAME}"))

    if allow_remote:
        candidates.append((model_name, False, MODEL_REVISION, f"hub:{model_name}"))
        candidates.append((FALLBACK_MODEL_NAME, False, None, f"hub:{FALLBACK_MODEL_NAME}"))

    if not candidates:
        raise RuntimeError(
            "No encontré el modelo en cache local. Coloca los pesos en HF_HOME o activa ALLOW_REMOTE_MODEL_DOWNLOAD=1 junto con HF_HUB_OFFLINE=0 para permitir descarga."
        )

    errors: list[str] = []
    for candidate, local_only, rev, label in candidates:
        try:
            model = SentenceTransformer(
                candidate,
                cache_folder=str(HF_HOME),
                trust_remote_code=True,
                local_files_only=local_only or offline,
                revision=rev,
                device=DEVICE
                            )
            try:
                model[0].auto_model.config.use_cache = False  # evita DynamicCache.get_usable_length
            except Exception:
                pass
            if label.startswith("cache:") and FALLBACK_MODEL_NAME in label:
                print(f"Usando fallback local {candidate}")
            return model
        except Exception as err:
            errors.append(f"{label} -> {err}")
            continue

    msg = "No pude cargar ningún encoder. Intentos:\n- " + "\n- ".join(errors)
    if not allow_remote:
        msg += "\nActiva ALLOW_REMOTE_MODEL_DOWNLOAD=1 y HF_HUB_OFFLINE=0 para permitir descarga."
    raise RuntimeError(msg)



def encode_reviews(texts: pd.Series, model: SentenceTransformer | None = None, batch_size: int = 256, show_progress: bool = True, device: str = DEVICE) -> np.ndarray:
    model = model or load_encoder()
    embeddings = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start : start + batch_size].tolist()
        try:
            model = model.to(device)
        except Exception:
            pass
        batch_emb = model.encode(
            batch,
            batch_size=batch_size,
            show_progress_bar=show_progress,
            normalize_embeddings=True,
            convert_to_numpy=True,
            device=device,
        )
        embeddings.append(batch_emb.astype("float32", copy=False))
    return np.vstack(embeddings)



def reduce_embeddings(x: np.ndarray, n_neighbors: int = 30, n_components: int = 15, metric: str = "cosine", random_state: int = RANDOM_STATE):
    reducer = umap.UMAP(
        n_neighbors=n_neighbors,
        n_components=n_components,
        metric=metric,
        random_state=random_state,
    )
    return reducer, reducer.fit_transform(x)



def cluster_embeddings(x: np.ndarray, min_cluster_size: int = 50, metric: str = "euclidean"):
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=min_cluster_size,
        metric=metric,
        cluster_selection_method="leaf",
    )
    return clusterer, clusterer.fit_predict(x)



In [5]:
# import torch, time

# model = load_encoder()
# print("target_device:", model._target_device)
# print("first param device:", next(model.parameters()).device)

# # mini‑benchmark para ver si usa GPU
# texts = ["hello world"] * 256
# t0 = time.perf_counter()
# _ = model.encode(texts, batch_size=128, device="cuda")
# torch.cuda.synchronize()
# print("encode(256) segs:", time.perf_counter() - t0)


## 3. Embeddings de las reviews
Calculamos (o cargamos) los embeddings. Si el `.npy` ya existe, se reutiliza para no recalcular. Se guardan también en el DataFrame para exportar a pickle.


In [ ]:
reviews = df["review"]
model = load_encoder(MODEL_NAME)
expected_dim = model.get_sentence_embedding_dimension()


def compute_and_store_embeddings():
    emb = encode_reviews(reviews, model=model, batch_size=256)
    emb = emb.astype("float32", copy=False)
    np.save(EMB_PATH, emb)
    print(f"Calculados y guardados embeddings en {EMB_PATH} -> {emb.shape}")
    return emb


if Path(EMB_PATH).exists():
    embeddings = np.load(EMB_PATH)
    if embeddings.shape[1] != expected_dim:
        print(
            f"Embeddings existentes {EMB_PATH} ({embeddings.shape[1]} dims) no coinciden con {expected_dim}; recalculando."
        )
        embeddings = compute_and_store_embeddings()
    else:
        print(f"Cargados embeddings desde {EMB_PATH} -> {embeddings.shape}")
else:
    embeddings = compute_and_store_embeddings()

embeddings = embeddings.astype("float32", copy=False)

df["embedding"] = embeddings.tolist()
df.to_pickle(DF_EMB_PATH)
df.head()


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

## 4. Reducción de dimensionalidad con UMAP
UMAP comprime los embeddings para que HDBSCAN trabaje en un espacio más denso. Guardamos tanto el modelo como la proyección para reutilizarla.


In [ ]:
if Path(EMB_UMAP_PATH).exists() and Path(UMAP_PATH).exists():
    umap_model = pickle.load(open(UMAP_PATH, "rb"))
    embeddings_umap = np.load(EMB_UMAP_PATH)
    print(f"UMAP cargado desde disco -> {embeddings_umap.shape}")
else:
    umap_model, embeddings_umap = reduce_embeddings(
        embeddings,
        n_neighbors=30,
        n_components=15,
        metric="cosine",
        random_state=RANDOM_STATE,
    )
    pickle.dump(umap_model, open(UMAP_PATH, "wb"))
    np.save(EMB_UMAP_PATH, embeddings_umap)
    print(f"UMAP entrenado y guardado -> {embeddings_umap.shape}")

embeddings_umap.shape


: 

## 5. Clustering con HDBSCAN
HDBSCAN detecta clusters de densidad variable y marca como `-1` el ruido. Ajusta `min_cluster_size` si quieres más/menos granularidad.


In [ ]:
clusterer, cluster_labels = cluster_embeddings( #clustering herarquico
    embeddings_umap,
    min_cluster_size=100, # cantidad de temas (clusters) mínimos
    metric="euclidean",
)

df["cluster_id"] = cluster_labels
pickle.dump(clusterer, open(CLUSTER_MODEL_PATH, "wb"))
df.to_pickle(DF_CLUSTER_PATH)

cluster_counts = df["cluster_id"].value_counts().reset_index()
cluster_counts.columns = ["cluster_id", "size"]
cluster_counts.head()


: 

## 6. Etiquetado legible de cada cluster
Para traducir IDs numéricos a nombres descriptivos:
- Calculamos TF-IDF por cluster (unigrams y bigrams) y extraemos las palabras más representativas.
- Creamos `cluster_label` con esos términos; para `-1` indicamos que es ruido/mixto.
- Generamos un catálogo con tamaño y un ejemplo de review por cluster. Así ves rápidamente qué significa cada número.


In [ ]:
def build_cluster_labels(df: pd.DataFrame, text_col: str = "review", cluster_col: str = "cluster_id", top_n: int = 5) -> dict:
    vectorizer = TfidfVectorizer(
        max_features=8000,
        stop_words="english",
        ngram_range=(1, 2),
        min_df=2,
    )
    tfidf = vectorizer.fit_transform(df[text_col])
    vocab = np.array(vectorizer.get_feature_names_out())
    labels = df[cluster_col].values
    cluster_map: dict[int, str] = {}
    for cid in np.unique(labels):
        mask = labels == cid
        scores = tfidf[mask].sum(axis=0).A1
        top_idx = scores.argsort()[::-1][:top_n]
        keywords = [vocab[i] for i in top_idx if scores[i] > 0][:top_n]
        label = ", ".join(keywords) if keywords else "general"
        if cid == -1:
            label = f"ruido/mixed — {label}"
        cluster_map[int(cid)] = label
    return cluster_map


def build_cluster_catalog(df: pd.DataFrame, cluster_label_col: str = "cluster_label") -> pd.DataFrame:
    return (
        df.groupby(["cluster_id", cluster_label_col])
        .agg(
            size=("review", "count"),
            example_review=("review", "first"),
        )
        .reset_index()
        .sort_values("size", ascending=False)
    )


cluster_name_map = build_cluster_labels(df)
df["cluster_label"] = df["cluster_id"].map(cluster_name_map)
cluster_catalog = build_cluster_catalog(df)
cluster_catalog.head(20)


: 

### Catálogo completo (id -> etiqueta -> tamaño -> ejemplo)
Ejecuta la siguiente celda para ver todas las categorías y qué significa cada número. Ajusta `head()` si quieres limitar.


In [ ]:
pd.set_option("display.max_rows", None)
# cluster_catalog
unique_types = cluster_catalog["cluster_id"].dropna().unique()
len(unique_types), unique_types


: 

## 7. Persistencia de artefactos
Guardamos embeddings, proyección UMAP, modelo HDBSCAN y DataFrame enriquecido para reutilizar sin recalcular.


In [ ]:
np.save(EMB_PATH, embeddings)
np.save(EMB_UMAP_PATH, embeddings_umap)
pickle.dump(umap_model, open(UMAP_PATH, "wb"))
pickle.dump(clusterer, open(CLUSTER_MODEL_PATH, "wb"))
df.to_pickle(DF_CLUSTER_PATH)


: 

## 8. Índice FAISS para búsqueda semántica
Normalizamos los embeddings y creamos un índice `IndexFlatIP` (similaridad coseno). Se guarda en disco para consultas rápidas.


In [ ]:
def build_faiss_index(x: np.ndarray) -> faiss.IndexFlatIP:
    x = np.asarray(x, dtype="float32", order="C")
    faiss.normalize_L2(x)
    index = faiss.IndexFlatIP(x.shape[1])
    index.add(x)
    return index

index = build_faiss_index(embeddings)
faiss.write_index(index, FAISS_INDEX_PATH)


: 

## 9. Búsqueda de reviews similares
Función de consulta que devuelve las reviews más cercanas junto con su `cluster_id` y `cluster_label` para interpretar el resultado.


In [ ]:
def search_reviews(user_text: str, top_k: int = 10, df_source: pd.DataFrame = df, model_ref: SentenceTransformer = model, index_ref = index):
    query_emb = model_ref.encode(
        [user_text],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32", copy=False)
    faiss.normalize_L2(query_emb)
    scores, idxs = index_ref.search(query_emb, k=top_k)
    return df_source.iloc[idxs[0]][["product_code", "review","review_stars", "cluster_id", "cluster_label"]]

user_text = "I’m looking for a scary unicorn."
results = search_reviews(user_text, top_k=10)
results


: 

## SIGUIENTES SON SOLO PROPUESTAS

## 10. Búsqueda two-stage por clusters + fallback a ruido
Estrategia para acelerar: primero se escogen los clusters más cercanos vía centroides, luego se busca solo en sus reviews. Si no alcanza `top_k`, se completa con el cluster de ruido (`-1`).

Flujo:
1) Precomputar centroides por `cluster_id` (excluyendo `-1`) y su índice FAISS pequeño.
2) Mantener un índice FAISS del ruido para fallback.
3) Consulta: seleccionar N clusters más cercanos, buscar en ese subconjunto, y si falta, completar con ruido.


In [ ]:
# Precomputar estructuras para búsqueda two-stage (centroides + ruido)
valid_df = df[df.cluster_id != -1].reset_index()
embs = embeddings  # alias corto

cluster_to_idx = (
    valid_df.groupby("cluster_id")["index"]
    .apply(list)
    .to_dict()
)

cluster_centroids = {
    cid: embs[idxs].mean(axis=0).astype("float32")
    for cid, idxs in cluster_to_idx.items()
}
centroid_ids = list(cluster_centroids.keys())
centroid_matrix = (
    np.vstack(list(cluster_centroids.values())).astype("float32")
    if centroid_ids else np.zeros((0, embs.shape[1]), dtype="float32")
)
faiss.normalize_L2(centroid_matrix)
centroid_index = faiss.IndexFlatIP(centroid_matrix.shape[1])
if len(centroid_ids) > 0:
    centroid_index.add(centroid_matrix)

# Índice de ruido (-1) para fallback
noise_idx = df.index[df.cluster_id == -1].to_numpy()
if len(noise_idx) > 0:
    embs_noise = embs[noise_idx].astype("float32", copy=True)
    faiss.normalize_L2(embs_noise)
    noise_index = faiss.IndexFlatIP(embs_noise.shape[1])
    noise_index.add(embs_noise)
else:
    noise_index = None


: 

In [ ]:
def search_two_stage_with_noise(
    user_text: str,
    top_clusters: int = 3,
    top_k: int = 10,
) -> pd.DataFrame:
    # Busca primero en los clusters más cercanos (por centroides) y luego en ruido si falta.
    if len(centroid_ids) == 0:
        return search_reviews(user_text, top_k=top_k)

    query_emb = model.encode(
        [user_text],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32", copy=False)
    faiss.normalize_L2(query_emb)

    # 1) Elegir clusters más cercanos
    _, c_idxs = centroid_index.search(query_emb, k=min(top_clusters, len(centroid_ids)))
    candidate_rows: list[int] = []
    for cid in [centroid_ids[i] for i in c_idxs[0]]:
        candidate_rows.extend(cluster_to_idx[cid])

    if not candidate_rows:
        return search_reviews(user_text, top_k=top_k)

    # 2) Búsqueda en subconjunto de clusters seleccionados
    subset = np.array(candidate_rows, dtype=int)
    sub_embs = embs[subset].astype("float32", copy=True)
    faiss.normalize_L2(sub_embs)
    sub_index = faiss.IndexFlatIP(sub_embs.shape[1])
    sub_index.add(sub_embs)
    _, s_idxs = sub_index.search(query_emb, k=min(top_k, len(subset)))
    hits = subset[s_idxs[0]]

    # 3) Fallback al ruido si faltan resultados
    if len(hits) < top_k and noise_index is not None and len(noise_idx) > 0:
        need = top_k - len(hits)
        _, n_idxs = noise_index.search(query_emb, k=min(need, len(noise_idx)))
        hits = np.concatenate([hits, noise_idx[n_idxs[0]]])

    return df.iloc[hits][["product_code", "review", "review_stars", "cluster_id", "cluster_label"]]


fast_results = search_two_stage_with_noise(
    "I’m looking for a cute unicorn.",
    top_clusters=3,
    top_k=10,
)
fast_results


: 

## 11. Búsqueda rápida: cluster único
Variante más veloz: se elige el cluster más cercano por centroides y se busca solo dentro de ese cluster con FAISS. Mayor rapidez, menor cobertura si la query cae entre temas.


In [ ]:
def search_cluster_first(user_text: str, top_k: int = 10) -> pd.DataFrame:
    # Elige el cluster más parecido (por centroides) y busca solo dentro de él.
    if len(centroid_ids) == 0:
        return search_reviews(user_text, top_k=top_k)

    query_emb = model.encode(
        [user_text],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32", copy=False)
    faiss.normalize_L2(query_emb)

    # 1) Cluster más cercano
    _, c_idxs = centroid_index.search(query_emb, k=1)
    best_cid = centroid_ids[int(c_idxs[0][0])]
    candidate_rows = cluster_to_idx.get(best_cid, [])
    if not candidate_rows:
        return search_reviews(user_text, top_k=top_k)

    # 2) Búsqueda dentro de ese cluster
    subset = np.array(candidate_rows, dtype=int)
    sub_embs = embs[subset].astype("float32", copy=True)
    faiss.normalize_L2(sub_embs)
    sub_index = faiss.IndexFlatIP(sub_embs.shape[1])
    sub_index.add(sub_embs)
    _, s_idxs = sub_index.search(query_emb, k=min(top_k, len(subset)))
    hits = subset[s_idxs[0]]

    return df.iloc[hits][["product_code", "review", "review_stars", "cluster_id", "cluster_label"]]

# Ejemplo de uso
fast_cluster_results = search_cluster_first(
    "I’m looking for a cute unicorn.",
    top_k=10,
)
fast_cluster_results


: 